In [0]:
CREATE OR REPLACE TEMP VIEW gift_rejoin_base AS
WITH created AS (
    SELECT cc_user_profile_id, plan_id, reporting_datetime,
           ROW_NUMBER() OVER (PARTITION BY cc_user_profile_id ORDER BY reporting_datetime, plan_id) AS created_seq,
           LEAD(plan_id) OVER (PARTITION BY cc_user_profile_id ORDER BY reporting_datetime, plan_id) AS next_plan_id,
           LEAD(reporting_datetime) OVER (PARTITION BY cc_user_profile_id ORDER BY reporting_datetime, plan_id) AS next_created_datetime
    FROM (
        SELECT DISTINCT cc_user_profile_id, plan_id, reporting_datetime
        FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
        WHERE event_type = 'subscription_created'
          AND cc_user_profile_id IS NOT NULL
          AND reporting_datetime IS NOT NULL
    )
),
first_gift AS (
    SELECT *
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY cc_user_profile_id ORDER BY reporting_datetime) AS gift_seq
        FROM created
        WHERE plan_id = 'gift-plan'
    )
    WHERE gift_seq = 1
),
gift_end AS (
    SELECT g.cc_user_profile_id,
           MAX(CAST(e.current_term_end_datetime AS DATE)) AS gift_end_date
    FROM first_gift AS g
    INNER JOIN cpx_dataanalytics_gold.marketing.mkt_subscription_events AS e
        ON e.cc_user_profile_id = g.cc_user_profile_id
    WHERE e.plan_id = 'gift-plan'
      AND e.event_type IN ('subscription_created','subscription_renewed','subscription_resumed','subscription_started')
      AND CAST(e.current_term_start_datetime AS DATE) >= DATE_SUB(CAST(g.reporting_datetime AS DATE), 1)
      AND (g.next_created_datetime IS NULL OR e.current_term_start_datetime < g.next_created_datetime - INTERVAL 1 DAY)
    GROUP BY g.cc_user_profile_id
)
SELECT g.cc_user_profile_id,
       CAST(g.reporting_datetime AS DATE)      AS gift_join_date,
       CASE WHEN g.created_seq = 1 THEN 1 ELSE 0 END AS first_ever_join,
       ge.gift_end_date,
       g.next_plan_id                          AS rejoin_plan_id,
       CAST(g.next_created_datetime AS DATE)   AS rejoin_date,
       DATEDIFF(CAST(g.next_created_datetime AS DATE), ge.gift_end_date) AS days_end_to_rejoin
FROM first_gift AS g
LEFT JOIN gift_end AS ge ON ge.cc_user_profile_id = g.cc_user_profile_id;


-- 2a. Grain + coverage: rows should equal unique members
SELECT COUNT(*)                                        AS rows_in_base,
       COUNT(DISTINCT cc_user_profile_id)              AS unique_members,
       SUM(first_ever_join)                            AS first_ever_join_members,
       COUNT_IF(gift_end_date IS NULL)                 AS missing_gift_end,
       COUNT_IF(gift_end_date >= current_date())       AS gift_still_active,
       COUNT_IF(gift_end_date <  current_date())       AS gift_ended
FROM gift_rejoin_base;


-- 2b. Timing of rejoin vs gift end (negative = rejoined before gift ended)
SELECT CASE WHEN rejoin_date IS NULL          THEN '0. No rejoin'
            WHEN days_end_to_rejoin < 0       THEN '1. Before gift end'
            WHEN days_end_to_rejoin <= 90     THEN '2. 0-90 days'
            WHEN days_end_to_rejoin <= 365    THEN '3. 91-365 days'
            ELSE '4. 365+ days' END           AS timing_bucket,
       rejoin_plan_id,
       COUNT(*)                               AS members
FROM gift_rejoin_base
WHERE gift_end_date < current_date()
GROUP BY 1, 2
ORDER BY 1, 2;


-- 2c. Gift members who moved to a paid plan via a plan change (no new signup event)
SELECT changed_to, COUNT(DISTINCT cc_user_profile_id) AS members
FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
WHERE event_type = 'subscription_changed' AND changed = TRUE
  AND changed_from = 'gift-plan' AND changed_from != changed_to
GROUP BY changed_to;


-- 2d. Eyeball sample
SELECT * FROM gift_rejoin_base ORDER BY gift_join_date DESC LIMIT 100;


SELECT COUNT_IF(rejoin_plan_id = 'gift-plan')  AS rejoined_as_gift_plan,
       COUNT_IF(rejoin_plan_id = 'annual1')    AS rejoined_as_annual,
       COUNT_IF(rejoin_plan_id = 'monthly1')   AS rejoined_as_monthly,
       COUNT_IF(rejoin_plan_id IS NOT NULL)    AS rejoined,
       COUNT_IF(rejoin_plan_id IS NULL)        AS never_rejoined,
       COUNT(*)                                AS unique_total,
       ROUND(100.0 * COUNT_IF(rejoin_plan_id IS NOT NULL) / COUNT(*), 1) AS pct_rejoined
FROM gift_rejoin_base
WHERE gift_end_date < current_date();


SELECT CONCAT(3 * n - 2, ' to ', 3 * n, ' months')          AS rejoin_window,
       COUNT_IF(in_window AND rejoin_plan_id = 'gift-plan') AS rejoined_as_gift_plan,
       COUNT_IF(in_window AND rejoin_plan_id = 'annual1')   AS rejoined_as_annual,
       COUNT_IF(in_window AND rejoin_plan_id = 'monthly1')  AS rejoined_as_monthly,
       COUNT_IF(in_window)                                  AS rejoined,
       COUNT(*)                                             AS unique_total,
       ROUND(100.0 * COUNT_IF(in_window) / COUNT(*), 1)     AS pct_rejoined
FROM (
    SELECT b.*, w.n,
           COALESCE(
               b.rejoin_date <= add_months(b.gift_end_date, 3 * w.n)
               AND (w.n = 1 OR b.rejoin_date > add_months(b.gift_end_date, 3 * (w.n - 1))),
               FALSE
           ) AS in_window
    FROM gift_rejoin_base AS b
    CROSS JOIN (SELECT explode(sequence(1, 10)) AS n) AS w
    WHERE add_months(b.gift_end_date, 3 * w.n) <= current_date()
)
GROUP BY n
ORDER BY n;



WITH dash AS (
    SELECT DISTINCT cc_user_profile_id
    FROM cpx_dataanalytics_gold.marketing.mkt_subscription_events
    WHERE event_type IN ('subscription_created','subscription_renewed','subscription_resumed','subscription_started')
      AND plan_id = 'gift-plan'
      AND current_date() BETWEEN CAST(current_term_start_datetime AS DATE) AND CAST(current_term_end_datetime AS DATE)
)
SELECT CASE WHEN b.cc_user_profile_id IS NULL        THEN '1. Not in base (no gift signup event)'
            WHEN b.gift_end_date >= current_date()   THEN '2. In base, first gift still active'
            WHEN b.rejoin_plan_id = 'gift-plan'      THEN '3. In base, now on a later gift'
            WHEN b.gift_end_date IS NULL             THEN '4. In base, missing gift end'
            ELSE '5. In base, other' END             AS where_they_sit,
       COUNT(*)                                      AS members
FROM dash AS d
LEFT JOIN gift_rejoin_base AS b ON b.cc_user_profile_id = d.cc_user_profile_id
GROUP BY 1
ORDER BY 1;

%sql
WITH b AS (
    SELECT *,
           CASE WHEN rejoin_date IS NULL          THEN 99
                WHEN rejoin_date < gift_end_date  THEN 0
                ELSE LEAST(11, GREATEST(1, CAST(CEIL(MONTHS_BETWEEN(rejoin_date, gift_end_date) / 3) AS INT)))
           END AS n
    FROM gift_rejoin_base
    WHERE gift_end_date < current_date()
)
SELECT CASE n WHEN 0  THEN 'Before gift end'
              WHEN 11 THEN 'After 30 months'
              WHEN 99 THEN 'Never rejoined'
              ELSE CONCAT(3 * n - 2, ' to ', 3 * n, ' months') END  AS rejoin_window,
       COUNT_IF(rejoin_plan_id = 'gift-plan')                       AS rejoined_as_gift_plan,
       COUNT_IF(rejoin_plan_id = 'annual1')                         AS rejoined_as_annual,
       COUNT_IF(rejoin_plan_id = 'monthly1')                        AS rejoined_as_monthly,
       COUNT(*)                                                     AS members,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)           AS pct_of_total
FROM b
GROUP BY n
ORDER BY n;